## stream data generation

In [3]:
import random
from dataclasses import dataclass, field

BANK_HANDLES = ["oksbi", "okhdfcbank", "okicici", "okaxis", "ybl", "paytm", "okbizaxis"]
INDIAN_STATES = [
    "Maharashtra", "Karnataka", "Delhi", "Tamil Nadu", "West Bengal",
    "Gujarat", "Uttar Pradesh", "Telangana", "Rajasthan", "Kerala",
    "Punjab", "Haryana", "Madhya Pradesh", "Bihar", "Odisha",
]

# @dataclass is a decorator from Python's standard library (dataclasses module) that auto-generates boilerplate for classes that just hold data — __init__, __repr__, and __eq__ — based on the fields you declare.

@dataclass
class User:
    user_id: str
    upi: str
    state: str
    devices: list = field(default_factory=list)
    typical_spend: float = 500.0

def build_users(n: int) -> list:
    users = []
    for i in range(n):
        bank = random.choice(BANK_HANDLES)
        num_devices = random.randint(1, 3)
        users.append(User(
            user_id=f"user{i:04d}",
            upi=f"user{i:04d}@{bank}",
            state=random.choice(INDIAN_STATES),
            devices=[f"user{i:04d}_{j}" for j in range(num_devices)],
            typical_spend=round(random.lognormvariate(6.2, 0.7), 2),  # right-skewed spend per user, median ~e^6.2≈490
        ))
    return users

a=build_users(100)
print("sample")
for i in a[:5]:
    print(i)

sample
User(user_id='user0000', upi='user0000@okicici', state='Kerala', devices=['user0000_0'], typical_spend=470.73)
User(user_id='user0001', upi='user0001@oksbi', state='Gujarat', devices=['user0001_0', 'user0001_1', 'user0001_2'], typical_spend=1178.83)
User(user_id='user0002', upi='user0002@okbizaxis', state='Telangana', devices=['user0002_0', 'user0002_1'], typical_spend=320.92)
User(user_id='user0003', upi='user0003@paytm', state='Maharashtra', devices=['user0003_0', 'user0003_1'], typical_spend=419.99)
User(user_id='user0004', upi='user0004@okaxis', state='Telangana', devices=['user0004_0', 'user0004_1', 'user0004_2'], typical_spend=631.76)


In [14]:
MERCHANT_CATEGORIES = [
    "grocery", "food_delivery", "fuel", "travel", "entertainment",
    "utilities", "ecommerce", "healthcare", "education", "electronics",
    "fashion", "pharmacy",
]

@dataclass
class Merchant:
    merchant_id: str
    upi: str
    category: str

def build_merchants(n: int) -> list:
    merchants = []
    for i in range(n):
        bank = random.choice(BANK_HANDLES)
        merchants.append(Merchant(
            merchant_id=f"merchant{i:03d}",
            upi=f"merchant{i:03d}@{bank}",
            category=random.choice(MERCHANT_CATEGORIES),
        ))
    return merchants

b=build_merchants(20)
print("sample")
for i in b[:5]:
    print(i)

sample
Merchant(merchant_id='merchant000', upi='merchant000@oksbi', category='pharmacy')
Merchant(merchant_id='merchant001', upi='merchant001@paytm', category='entertainment')
Merchant(merchant_id='merchant002', upi='merchant002@ybl', category='fuel')
Merchant(merchant_id='merchant003', upi='merchant003@ybl', category='food_delivery')
Merchant(merchant_id='merchant004', upi='merchant004@okbizaxis', category='fashion')


In [22]:
random.seed(42)
user=build_users(20)
merchant=build_merchants(5)

reciever=random.choice(merchant)

def select_receiver(sender,user,merchant):
    ## 60% merchants and 40% person 2 person
    if random.random() <0.6:
        m=random.choice(merchant)
        return m.merchant_id,"P2M",m.category
    u=random.choice(user)
    while sender.upi==u.upi:
        u=random.choice(user)
    return u.upi,"P2P",None

for i in range(5):
    sender=random.choice(user)
    receiver_upi,payment_segment,receiver_category=select_receiver(sender,user,merchant)
    print(f"mode: {i}")
    print("sender details: " ,sender.user_id, " | ",sender.upi," | ",sender.state," | ",sender.typical_spend,random.choice(sender.devices))
    print("receiver details: " ,receiver_upi, " | ",payment_segment," | ",receiver_category)

mode: 0
sender details:  user0019  |  user0019@ybl  |  Gujarat  |  498.99 user0019_1
receiver details:  user0016@ybl  |  P2P  |  None
mode: 1
sender details:  user0017  |  user0017@paytm  |  Delhi  |  338.74 user0017_0
receiver details:  user0000@paytm  |  P2P  |  None
mode: 2
sender details:  user0017  |  user0017@paytm  |  Delhi  |  338.74 user0017_0
receiver details:  user0010@oksbi  |  P2P  |  None
mode: 3
sender details:  user0009  |  user0009@okbizaxis  |  Kerala  |  1156.75 user0009_0
receiver details:  merchant003  |  P2M  |  electronics
mode: 4
sender details:  user0008  |  user0008@okaxis  |  Rajasthan  |  252.45 user0008_0
receiver details:  user0005@okaxis  |  P2P  |  None


In [26]:
fraud_rate=0.2

n=10
f=int(n*fraud_rate)
nf=n-f

import uuid
from datetime import datetime,timezone

def make_event(sender, receiver_upi, receiver_type, receiver_category,
               amount, device_id, status, is_fraud=False, fraud_pattern=None,
               timestamp=None) -> dict:
    ts = datetime.now(timezone.utc)
    return {
        "txn_id": str(uuid.uuid4()),
        "timestamp": ts.isoformat(timespec="seconds").replace("+00:00", "Z"),
        "sender_upi": sender.upi,
        "sender_state": sender.state,
        "sender_device_id": device_id,
        "receiver_upi": receiver_upi,
        "receiver_type": receiver_type,
        "receiver_category": receiver_category,
        "amount": round(amount, 2),
        "status": status,
        "is_fraud": is_fraud,
        "fraud_pattern": fraud_pattern,
    }

op=[]
for i in range(nf):
    sender=random.choice(user)
    receiver_upi,payment_segment,receiver_category=select_receiver(sender,user,merchant)
    op.append(make_event(sender,receiver_upi,payment_segment,receiver_category,sender.typical_spend,"SUCCESS","N",""))

# fraud event
for i in range(f):
    sender=random.choice(user)
    receiver_upi,payment_segment,receiver_category=select_receiver(sender,user,merchant)
    amt=round(random.randint(10000,100000),2)
    op.append(make_event(sender,receiver_upi,payment_segment,receiver_category,amt,"SUCCESS","Y","high_spend"))

op


[{'txn_id': 'a5363395-8e4e-49fb-8771-0bf2a256143b',
  'timestamp': '2026-07-22T16:40:01Z',
  'sender_upi': 'user0001@okhdfcbank',
  'sender_state': 'Karnataka',
  'sender_device_id': 'SUCCESS',
  'receiver_upi': 'user0003@ybl',
  'receiver_type': 'P2P',
  'receiver_category': None,
  'amount': 3526.36,
  'status': 'N',
  'is_fraud': '',
  'fraud_pattern': None},
 {'txn_id': 'bab539bd-893b-4832-b4cd-5a4c4cf3bf1c',
  'timestamp': '2026-07-22T16:40:01Z',
  'sender_upi': 'user0001@okhdfcbank',
  'sender_state': 'Karnataka',
  'sender_device_id': 'SUCCESS',
  'receiver_upi': 'merchant002',
  'receiver_type': 'P2M',
  'receiver_category': 'fashion',
  'amount': 3526.36,
  'status': 'N',
  'is_fraud': '',
  'fraud_pattern': None},
 {'txn_id': 'd5f68e48-8d6a-45c8-ad19-910015595776',
  'timestamp': '2026-07-22T16:40:01Z',
  'sender_upi': 'user0003@ybl',
  'sender_state': 'Haryana',
  'sender_device_id': 'SUCCESS',
  'receiver_upi': 'merchant001',
  'receiver_type': 'P2M',
  'receiver_category':

In [ ]:
x=[]

def high_spend(users, merchants):
    sender=random.choice(users)
    receiver_upi,payment_segment,receiver_category=select_receiver(sender,user,merchants)
    amt=round(random.randint(10000,100000),2)    
    return make_event(sender,receiver_upi,payment_segment,receiver_category,amt,"SUCCESS","Y","high_spend")


def odd_hours(users, merchants):
    sender=random.choice(users)
    receiver_upi,payment_segment,receiver_category=select_receiver(sender,users,merchants)
    odd_hour_ts = datetime.now(timezone.utc).replace(
        hour=random.randint(1, 4), minute=random.randint(0, 59), second=random.randint(0, 59),
    )
    amt=sender.typical_spend*random.uniform(8,10)
    return make_event(sender,receiver_upi,payment_segment,receiver_category,amt,"SUCCESS","Y","odd_hours",timestamp=odd_hour_ts)

def legit_txn(users, merchants):
    sender=random.choice(users)
    receiver_upi,payment_segment,receiver_category=select_receiver(sender,user,merchants)
    return make_event(sender,receiver_upi,payment_segment,receiver_category,sender.typical_spend,"SUCCESS","N","")





In [ ]:
n=30
fraud_rate=0.5
f=int(n*fraud_rate)
nf=round(n-f)

txn=[]
for _ in range(nf):
    txn.append(legit_txn(user,merchant))

frd=0
n_frd=0
while frd+n_frd<n:
    if frd<n:
        generator=random.choice([legit_txn,odd_hours,high_spend])
        
        if generator==odd_hours:
            frd+=1
            txn.append(generator(user,merchant))
        else:
            txn.append(generator(user,merchant))
            n_frd+=1

        continue
    else:
        txn.append(high_spend(user,merchant))
        n_frd+=1

    

import pandas as pd

df=pd.DataFrame(txn)
df


In [ ]:
## version 2 fraud samples

n=30
fraud_rate=0.5
f=int(n*fraud_rate)
nf=round(n-f)

def rapid_burst(users, merchants,rb) -> list:
    sender=random.choice(users)
    receiver_upi,payment_segment,receiver_category=select_receiver(sender,users,merchants)
    
    amt=random.randint(1000,10000)
    rapid_op=[]
    for _ in range(rb):
        rapid_op.append(make_event(sender,receiver_upi,payment_segment,receiver_category,amt,"SUCCESS","Y","rapid_burst"))

    return rapid_op

txn=[]
for _ in range(nf):
    txn.append(legit_txn(user,merchant))

cnt=f
while cnt<n:
    rb=random.randint(3,5)
    if rb>n-cnt:
        generator=random.choice([rapid_burst,high_spend])
        
        if generator==rapid_burst:
            cnt+=rb
            txn.extend(generator(user,merchant,rb))
        else:
            txn.append(high_spend(user,merchant))
            cnt+=1

        continue
    else:
        txn.append(high_spend(user,merchant))
        cnt+=1

    

txn       


In [ ]:
def pick_receiver(sender, users, merchants):
    """60% P2M / 40% P2P, matching legitimate traffic mix."""
    if random.random() < 0.6:
        merchant = random.choice(merchants)
        return merchant.upi, "P2M", merchant.category
    other = random.choice(users)
    while other.user_id == sender.user_id:
        other = random.choice(users)
    return other.upi, "P2P", None




In [9]:
STATUS_CHOICES = ["SUCCESS", "FAILED", "REVERSED"]
STATUS_WEIGHTS = [0.95, 0.04, 0.01]

random.choices(STATUS_CHOICES, weights=STATUS_WEIGHTS, k=1)[0]

'SUCCESS'

## Pydantic data model

In [ ]:
from pydantic import BaseModel, field_validator, model_validator, ConfigDict

class Event(BaseModel):
    model_config = ConfigDict(strict=True)  # no silent coercion

    txn_id: int
    email: str
    name: str

    @field_validator("txn_id")
    @classmethod
    def txn_id_must_be_positive(cls, v: int) -> int:
        if v <= 0:
            raise ValueError("txn_id must be a positive integer") 
        return v

    @field_validator("email")
    @classmethod
    def email_must_be_valid(cls, v: str) -> str:
        if "@" not in v or "." not in v.split("@")[-1]:
            raise ValueError(f"'{v}' is not a valid email address") # raise exception 
        return v.lower().strip()                                    # return after data transformation lower and strip 

    @field_validator("name")
    @classmethod
    def name_must_not_be_empty(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("name cannot be empty or whitespace")
        return v.title()  # "raj" -> "Raj"

    @model_validator(mode="after")
    def cross_field_check(self):
        # runs after all field validators pass — for rules spanning multiple fields
        if self.name.lower() in self.email:
            pass  # example hook — e.g. flag suspicious matches, log, etc.
        return self

    @model_validator(mode="before")
    def cross_field_check_before(self):
            # runs before field validators — e.g. reject if required raw fields are missing
            if not self.name or not self.email:
                pass  # example hook — e.g. flag suspicious matches, log, etc.
            return self

# a=Event(txn_id=1234,email="raj.guru@gmail",name="raj" ) # pass
# a=Event(txn_id="1234",email="raj.guru@gmail.com",name="raj" ) # fails due to txn_id datatype err
# a=Event(txn_id=-1234,email="raj.guru@gmail.com",name="raj" ) # fails due to txn_id <0
# a=Event(txn_id=1234,email="raj.guru@gmail.com",name="raju" ) # fails due invalid cross validation name not in email
# a=Event(txn_id=1234,email="raj.guru@gmail.com" ) # fails since name is None
# a=Event(txn_id=1234,email="raj.guru@gmail",name="raj" ) # Fail due to invalid email

a=Event(txn_id=1234,email="raj.guru@gmail.com",name="raj" )

In [ ]:
## data clean before model build

from pydantic import BaseModel, field_validator

class Event(BaseModel):
    txn_id: int

    @field_validator("txn_id", mode="before")
    @classmethod
    def clean_txn_id(cls, v):
        # v is RAW here — could be anything: str, int, float, None...
        if isinstance(v, str):
            v = v.strip().replace(",", "")  # "1,234 " -> "1234"
        return v

In [ ]:
# reshape raw input

from pydantic import BaseModel, model_validator

class Event(BaseModel):
    txn_id: int
    email: str

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        # Some upstream source sends "transaction_id" instead of "txn_id"
        if isinstance(data, dict):
            if "transaction_id" in data and "txn_id" not in data:
                data["txn_id"] = data.pop("transaction_id")
        return data

Event(transaction_id=101, email="raj@gmail.com")
# txn_id=101 email='raj@gmail.com'

### pydantic in Spark


Why not (in distributed transforms): Spark's engine works in JVM/columnar space (DataFrame, StructType) and has no concept of a Pydantic object. Using Pydantic per-row means shipping every row into Python, instantiating an object, validating, serializing back — that per-row overhead kills parallelism at scale (millions of rows).

Where it does work:

Config/schema validation — validate YAML/job parameters once before the cluster even spins up.
Small-batch/API payload validation — clean incoming JSON before spark.createDataFrame().
UDF/pandas UDF — usable, but only for small volumes; not a scale solution.

Rule of thumb: Pydantic validates config and inputs around Spark; DLT expectations / native filter() / pandera validate data inside Spark.